# Véore — Free Colab Video Generator (LTX-Video 2B)

Make short **vertical product clips** (about 3 seconds, 9:16) on Google Colab's **free T4 GPU** using the open-source **LTX-Video 2B** model. No credits, no payment — you run it yourself, then send the clips back for editing/QC.

## How to use (4 clicks)
1. **Runtime → Change runtime type → T4 GPU → Save** (must be done first!)
2. **Runtime → Run all** — Steps 1–3 set everything up (Step 3 downloads the model, ~28 GB, only on the first run of a session).
3. Edit the **PROMPT** in **Step 4** and run just that cell again for each new clip.
4. **Step 5** shows a preview and downloads the MP4.

## Defaults
- Vertical **448 × 800**, **73 frames @ 24 fps ≈ 3 s**, **25 steps** — expect roughly **3–7 minutes** per clip on the free T4.
- Want 4–5 s? Set `NUM_FRAMES = 97` (≈4 s) or `121` (≈5 s) — slower and uses more memory.
- The model also does **image-to-video**: set `MODE = "image-to-video"` in Step 4 and upload a product photo as the first frame.


In [ ]:
# @title Step 1 — Check the GPU
import subprocess
import torch

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), (
    "No GPU found. Go to Runtime -> Change runtime type -> select 'T4 GPU' -> Save, "
    "then run this cell again."
)
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU OK: {name} - {vram:.1f} GB VRAM")


In [ ]:
# @title Step 2 — Install libraries (once per session, ~1-2 min)
# Colab already has PyTorch, so we only add the video-generation stack.
# (If Colab later asks you to restart the runtime, do it, then continue from Step 3.)
%pip install -q "diffusers==0.42.1" "accelerate==1.15.0" transformers sentencepiece "imageio[ffmpeg]" pillow
print("Install done. Continue to Step 3.")


In [ ]:
# @title Step 3 — Download & load the model (first run downloads ~28 GB)
import importlib.util
import subprocess
import sys

import torch
from diffusers import LTXPipeline
from transformers import T5EncoderModel, TorchAoConfig

MODEL_ID = "Lightricks/LTX-Video"  # LTX-Video 2B by Lightricks, via Hugging Face diffusers

# Optional: to avoid re-downloading 28 GB every new session, mount Google Drive first
# (left sidebar -> folder icon) and uncomment the next two lines BEFORE running this cell:
# import os
# os.environ["HF_HOME"] = "/content/drive/MyDrive/hf-cache"

# --- Why this cell changed (v2) ---------------------------------------------------
# v1 crashed here with "session crashed after using all available RAM".
# The download is ~28 GB and most of it is the T5-XXL text encoder, stored on the
# Hub in float32. Loading the whole pipeline in float16 still parks ~13-14 GB of
# weights in system RAM (free Colab has only ~13 GB), so the session died while
# the checkpoint shards were loading -- before CPU offload could even start.
#
# Fix: load the text encoder on its own, quantized to int8 WHILE it loads
# (TorchAO int8 weight-only, via the Hugging Face transformers integration).
# It ends up ~5 GB instead of ~9.5 GB, and no full-precision copy of it is ever
# held in RAM; the quality impact is negligible for short clips. The rest of the
# model (the 2B video transformer + VAE) is small enough to load normally in
# fp16. The same TorchAO recipe is what the diffusers quantization docs use on
# CogVideoX, which ships the very same T5-XXL encoder.
#
# Free-tier runtime settings (unchanged from v1):
# - torch_dtype=float16: the T4 has no fast bfloat16 support, so fp16 is right here.
# - enable_sequential_cpu_offload(): streams the model through the GPU piece by
#   piece at generation time, so it fits the T4's 16 GB VRAM. A little slower,
#   but it will not crash.
# - vae.enable_tiling(): decodes the video in tiles to save memory at the end.

# TorchAO ships separately from transformers; install it only if it is missing.
if importlib.util.find_spec("torchao") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchao"], check=True)

# 1) Text encoder first, int8-quantized during loading (~5 GB in RAM, not ~9.5 GB).
text_encoder = T5EncoderModel.from_pretrained(
    MODEL_ID,
    subfolder="text_encoder",
    quantization_config=TorchAoConfig("int8_weight_only"),
    torch_dtype=torch.float16,
)

# 2) Load the rest of the pipeline around it. The encoder above is reused as-is,
#    so it is not downloaded or loaded a second time.
pipe_t2v = LTXPipeline.from_pretrained(
    MODEL_ID,
    text_encoder=text_encoder,
    torch_dtype=torch.float16,
)
pipe_t2v.enable_sequential_cpu_offload()
pipe_t2v.vae.enable_tiling()

print("Model ready. Go to Step 4, edit the PROMPT, and run that cell.")


In [ ]:
# @title Step 4 — Generate a clip (edit settings, then run)
import random

import torch
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video
from PIL import Image

MODE = "text-to-video"  # @param ["text-to-video", "image-to-video"]
PROMPT = "A slim silver motion-sensor LED light bar mounted on the wall of a dark hallway. A person walks past in the background and the light bar switches on by itself, glowing warm white and lighting up the hallway floor. Static camera, realistic home video, smooth natural motion, photorealistic."  # @param {type:"string"}
NEGATIVE_PROMPT = "worst quality, inconsistent motion, blurry, jittery, distorted, text, watermark"  # @param {type:"string"}
INPUT_IMAGE_PATH = ""  # @param {type:"string"}
# ^ image-to-video only: path of the start image inside Colab (e.g. /content/start.jpg).
#   Leave empty and you will be asked to upload one.
WIDTH = 448  # @param {type:"integer"}
HEIGHT = 800  # @param {type:"integer"}
# ^ Vertical 9:16. Both numbers must be divisible by 32. For landscape, swap them.
NUM_FRAMES = 73  # @param {type:"integer"}
# ^ Must be 8k+1: 73 = ~3 s, 97 = ~4 s, 121 = ~5 s at 24 fps.
STEPS = 25  # @param {type:"integer"}
# ^ 20-30 is the speed/quality sweet spot on a free T4.
SEED = 42  # @param {type:"integer"}
# ^ Keep the same seed to reproduce a clip; set -1 for a random one.
FPS = 24  # @param {type:"integer"}
OUTPUT_PATH = "/content/veore_clip.mp4"  # @param {type:"string"}

if SEED == -1:
    SEED = random.randint(0, 2**31 - 1)
# A CPU generator is recommended together with CPU offloading.
generator = torch.Generator().manual_seed(SEED)

if MODE == "image-to-video":
    if not INPUT_IMAGE_PATH:
        from google.colab import files

        uploaded = files.upload()
        INPUT_IMAGE_PATH = next(iter(uploaded))
    start_image = Image.open(INPUT_IMAGE_PATH).convert("RGB").resize((WIDTH, HEIGHT))
    # Build the image-to-video pipeline from the SAME model parts (no second download).
    pipe = LTXImageToVideoPipeline(
        transformer=pipe_t2v.transformer,
        scheduler=pipe_t2v.scheduler,
        vae=pipe_t2v.vae,
        text_encoder=pipe_t2v.text_encoder,
        tokenizer=pipe_t2v.tokenizer,
    )
    result = pipe(
        image=start_image,
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        width=WIDTH,
        height=HEIGHT,
        num_frames=NUM_FRAMES,
        num_inference_steps=STEPS,
        decode_timestep=0.03,
        decode_noise_scale=0.025,
        generator=generator,
    )
else:
    result = pipe_t2v(
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        width=WIDTH,
        height=HEIGHT,
        num_frames=NUM_FRAMES,
        num_inference_steps=STEPS,
        decode_timestep=0.03,
        decode_noise_scale=0.025,
        generator=generator,
    )

frames = result.frames[0]
export_to_video(frames, OUTPUT_PATH, fps=FPS)
LAST_OUTPUT = OUTPUT_PATH
print(f"Saved: {OUTPUT_PATH} ({len(frames)} frames, {WIDTH}x{HEIGHT}, seed {SEED})")

# Troubleshooting:
# - "CUDA out of memory": lower WIDTH/HEIGHT (e.g. 384x672) or NUM_FRAMES (49),
#   then Runtime -> Restart session, and run Steps 3-4 again.
# - Blurry or messy motion: raise STEPS to 30-35, and make the PROMPT more detailed
#   (LTX works best with long, descriptive English prompts).


In [ ]:
# @title Step 5 — Preview & download the clip
from IPython.display import Video, display
from google.colab import files

try:
    path = LAST_OUTPUT
except NameError:
    path = "/content/veore_clip.mp4"

display(Video(path, embed=True, width=360))
files.download(path)
print("Downloaded:", path)


## Tips
- **Prompts:** LTX likes long, detailed English prompts. Describe the product, the action, the camera (e.g. "static camera", "slow push-in"), and the light.
- **One action per clip** works best (e.g. "the drawer slides out"), instead of many things happening at once.
- **More clips:** just re-run **Step 4** with a new prompt or seed — no need to re-run Steps 1–3.
- If a run fails with *out of memory*, lower the resolution to **384 × 672** or frames to **49**, restart the runtime, and run Steps 3–4 again.
